# Introduction
As of 2025-06-12, the volunteers at rangers.urbanrivers have added 59,351 observations.  
These observations are not error-proof, but they are definitely a useful means of creating base-truth casses for image labeling.  

This notebook serves as a means of aggregating those data from the server api in a scaleable manner - to where as observations continue to grow, we can ingest new labels for model improvement


In [2]:
# Data Handling
import pandas as pd

# IO - getting files and images
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient
import requests
import json
import os
import urllib.parse

# For randomizing which images get downloaded
import random
from tqdm.auto import tqdm

# For model loading and fine tuning
from fastai.vision.all import *
from fastai.vision.widgets import *
import torch

print("==== Loaded Libraries ====")

==== Loaded Libraries ====


# Accessing observations (image labels) from the public api
This version uses pymongo (MongoClient) 

In [7]:
%%time
# Get the stored mongo uri secret
user_secrets = UserSecretsClient()
mongo_uri = user_secrets.get_secret("MONGO_PROD")

# Access the server
client = MongoClient(mongo_uri)
db = client['test']
collection = db['cameratrapmedias']

# Fetch documents with at least one speciesConsensus entry
def fetch_all_obs():
    query = {"speciesConsensus.0": {"$exists": True}}  # at least one item
    projection = {
        "_id": 0,
        "mediaID": 1,
        "publicURL": 1,
        "speciesConsensus": 1
    }
    all_obs = list(collection.find(query, projection))
    print(f"Retrieved {len(all_obs)} documents with speciesConsensus.")
    return all_obs

# Try the fetch operation
try:
    print("===== Starting MongoDB Fetch =====")
    obs_json = fetch_all_obs()
except Exception as e:
    print(f"Error during fetch: {e}")

Total matching documents: 59999
===== Starting MongoDB Fetch =====
Retrieved 59999 documents with speciesConsensus.
CPU times: user 656 ms, sys: 144 ms, total: 800 ms
Wall time: 4.06 s


## Process the returned JSON for the fields we need
We're looking for the `mediaID` (our primary key),  
The `publicURL` that we can use to download the image,  
The `scientificName` that observers have selected and the `observationCount` of times that people have agreed on the species.  
  -  Note this is different than a similar field `count` that represents the number of species in the photo
  -  An `observationCount` increase requires both the species and the `count` to be the same.
  -  For example, a `count` of 1 canis familiaris with a `observationCount` of 1 could also have a `count` of 2 and `observationCount` of 3 if 3 people saw 2 dogs, and 1 person saw 1 dog.
  -  In version 5 we include the count

In [8]:
%%time
# Process the JSON into flat records
def process_obs_json(obs):
    records = []
    for ob in obs:
        media_id = ob.get("mediaID")
        public_url = ob.get("publicURL")
        for species in ob.get("speciesConsensus", []):
            records.append({
                "mediaID": media_id,
                "publicURL": public_url,
                "scientificName": species.get("scientificName"),
                "observationCount": species.get("observationCount"),
                "speciesCount": species.get("count")
            })
    return pd.DataFrame(records)

# Process the returned JSON
try:
    print("===== Starting Data Processing =====")
    df = process_obs_json(obs_json)

    with pd.option_context('display.width', 0, 'display.max_colwidth', None):
        display(df.head())

    print(f"Total records: {len(df)}")
except Exception as e:
    print(f"Error during processing: {e}")

===== Starting Data Processing =====


,mediaID,publicURL,scientificName,observationCount,speciesCount
0,c112813a5f3b9cec26f95fad982b8d09,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0001.JPG,None,1,1
1,0647380f2d59692f5b2b642312844e9f,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0002.JPG,None,1,1
2,0db73c6c1efb4968c04a47e418ebeefb,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0004.JPG,None,1,1
3,31fc53de29056b4dd8bc7b1804617f00,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0006.JPG,None,1,1
4,14664d764836c5fd9a38284dc6103527,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0008.JPG,None,1,1


Total records: 67691
CPU times: user 146 ms, sys: 3.3 ms, total: 149 ms
Wall time: 151 ms


## Save data as we go along for referencing later as needed
We're going to 'checkpoint' a few files while filtering because we might go back or try different types of classification or detection later


In [9]:
# Save the processed data as is
os.makedirs('/kaggle/working/data', exist_ok=True)
df.to_csv('/kaggle/working/data/initial_processed_data.csv', index=False)
df = pd.read_csv("/kaggle/working/data/initial_processed_data.csv")
print("\n==== Saved checkpoint 1 ====\n")


==== Saved checkpoint 1 ====



# Data Cleaning and Filtering
The raw data from the observations could have a few issues:  
Duplication - if there are bugs in the observation recording process
False positives - if people classify the wrong species in an image, or accidentally classify multiple when there is only one
False negatives - from time to time, people could be classifying long groups of blanks, and then accidentally skip an image with an animal

In [ ]:
# Solve the potential duplicates for every image and observation - just want a validated list
df2 = df.drop_duplicates().reset_index(drop=True)
with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(df2.head())

print(f'Total rows: {len(df2)}\n')
print(df2['scientificName'].value_counts())

## Counts vs ObsCounts
In version 5 of this notebook we added speciesCount - When deduplicating, now if there are votes for 1x and 2x of a species, there are potentially new rows.  
We'll maintain a minimum vote requirement of 3 - this should reduce times where only 1 or 2 people have voted incorrectly.

In [ ]:
# Filter to species with at least 3 votes
df3 = df2[(df2['observationCount'] >= 3) ]
df3.loc[:, 'scientificName'] = df3['scientificName'].fillna("blank")
df3 = df3.sort_values(by='observationCount', ascending=False).reset_index(drop=True)
with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(df3.head())

print(f'Total rows: {len(df3)}\n')
print(df3['scientificName'].value_counts())

You can tell now that our value_counts report is showing the full list of species because some more rare species haven't been confirmed by at least 3 people.  

## Identifying multiple classification images vs single classification
Animals share spaces.  
As long as there isn't animosity, it's very possible that some images have more than one species -  
For the purpose of model type, we need to delineate what we're fine-tuning to, so it's important to group and understand the nature of each image from the observations.

### Dropping speciesCount while grouping
The group by below directly references mediaID and publicURL - this effectively collapses the potential for images with 1 count of a species together with multiple counts.  
Another options here would be to include the speciesCount in the group by if we wanted to start classifying the number in addition to the type in images (we don't for now).  
ex:  
`df3_grouped = df3.groupby(['mediaID', 'publicURL', 'speciesCount'])['scientificName'] \ ...`

In [ ]:
# This might be a multiple classification problem - let's see if things change when we group by and list the scientific names
df3_grouped = df3.groupby(['mediaID', 'publicURL'])['scientificName'] \
    .agg(lambda x: ';'.join(sorted(set(x)))) \
    .reset_index()

# To keep classes where only blank and yet not ones containing blank
def is_only_blank(label_str):
    return label_str.strip() == "blank"

# Filter: keep rows where "blank" is not in the list OR is the only label
# This is because sometimes people are categorizing multiple blanks in a row and do not see the animal while zoned out.
df3_grouped = df3_grouped[
    ~df3_grouped['scientificName'].str.contains(';blank') |
    df3_grouped['scientificName'].apply(is_only_blank)
]

# Show the results
with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(df3_grouped.head())
    
print(f'Total rows: {len(df3_grouped)}\n')
print(df3_grouped['scientificName'].value_counts())

In [ ]:
# Save the filtered df as a starting point for image requests and multiclassification future decisions
df3_grouped.to_csv("/kaggle/working/data/all_labeled_species_urls.csv", index=False)
print("\n==== Saved checkpoint 2 ====\n")

### As of 2025-06-13 there are 1379 rows for the 36 classifications

There are a few that are classified as multiple species - for now, let's focus on those that are just single labeled.

In [ ]:
# We're going to focus on single species labeled images and blanks for an attempt at training a simple classification model - a scientific name always has at least one space - so we'll filter one last time.
df4 = df3_grouped[(~df3_grouped["scientificName"].str.contains(";")) & (df3_grouped["scientificName"].str.contains(" ")) | (df3_grouped["scientificName"] == 'blank')]

# Show the results
with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(df4.head())
    
print(f'Total rows: {len(df4)}\n')
print(df4['scientificName'].value_counts())

## Single species type in image subset - 
### As of 2025-06-13 there are 1228 rows for 21 classified species
There are some more rare species than others.  

Branta canadensis - the canadian goose - there are 556 images exist of that classification today, where..  
Castor canadensis - the beaver - only has 48 and ..  
Nycticorax nycticorax - the Night Heron - only has 4 (but we do have a lot of them in photos)  

Part of this is because people don't recognize some speces - lumping them into a higher taxonomic class, like 'aves'.  
For the purpose of having a good number of images for training we need to set a minimum threshold.  

For right now, we're going to use 26, but realistic production models might need more like 200.  
The decision to allow at least 26 is to have more classes for fine-tuning and testing:  
25 for training with a 20% validation (5 of 25) and 1 for testing later.

## Set a minimum number of observations so our model has some chance at learning

In [ ]:
# Adjust as the project progresses:
n_obs_minimum = 26

In [ ]:
# Filter current df to where the minimum is met
species_counts = df4['scientificName'].value_counts()
species_to_keep = species_counts[species_counts >= n_obs_minimum].index

df5 = df4[df4['scientificName'].isin(species_to_keep)]

# Show the results
with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(df5.head())
    
print(f'Total rows: {len(df5)}\n')
print(df5['scientificName'].value_counts())

### We're left with 1142 rows for 8 classifications

In [ ]:
# save the single species images with at least n observations as a data checkpoint
df5.to_csv(f'/kaggle/working/data/species_over{n_obs_minimum}.csv', index=False)
print("\n==== Saved checkpoint 3 ====\n")

# Downloading Images that meet the criteria
Passing our filtered df to the s3 bucket to request images in grouped train or test splits

In [ ]:
# Select the df cleaning state we want to use
df = df5

# Create the directories
base_dir = '/kaggle/working/images'
test_dir = '/kaggle/working/_tests'
os.makedirs(test_dir, exist_ok=True)

# Shuffle the whole DataFrame first to ensure randomness
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
grouped = list(df_shuffled.groupby('scientificName'))

In [ ]:
# Show the structure of the grouped list for understanding
print("speciesName: ", grouped[0][0])
print("dataframe: ")
grouped[0][1].head()

## Grouped is a list where each index species has a dataframe  so key, value is scientific_name, df of images

In [ ]:
# fast ai version of image downloading
from fastdownload import download_url

ims = grouped[0][1]['publicURL'].tolist()
imn = grouped[0][1]['mediaID'].tolist()

len(ims)

dest = os.path.join(base_dir, grouped[0][0].replace(' ','_'), f'{imn[0]}.jpg')
print(dest)

download_url(ims[0], dest)

im = Image.open(dest)
im.to_thumb(244,244)

In [ ]:
# For cleanup while testing
!rm /kaggle/working/images/* -rf

In [ ]:
%%time
# fast ai version of all images downloading - drops the mediaID from the pipeline which might be ok
for species, group in grouped:
    group = group.dropna(subset=['publicURL'])
    if len(group) < n_obs_minimum:
        continue

    selected = group.sample(n=n_obs_minimum, random_state=42).reset_index(drop=True)
    n_split = n_obs_minimum - 1
    train_samples = selected.iloc[:n_split]
    test_sample = selected.iloc[n_split]

    train_urls = train_samples['publicURL'].tolist()
    test_url = test_sample['publicURL']
    
    
    species_folder = species.replace(' ', '_')
    species_dir = os.path.join(base_dir, species_folder)
    
    download_images(species_dir, urls=train_urls)
    print(species_dir)
    
    # Download test image
    test_image_path = os.path.join(test_dir, f'{species_folder}.JPG')
    download_url(test_url, test_image_path)

print("==== Downloaded Images ====")
    

In [ ]:
# validate images
fns = get_image_files(base_dir)
failed = verify_images(fns)
print(failed)
failed.map(Path.unlink);

# For each image downloaded we are going to interact with fastai
Dataloaders (dls) are tensors that fastai simplifies for use in model fine-tuning.  
The library loads only what's needed (even though we passed * at the top) and has some built in common features.  

We're planning on using resnet18 - so we want to resize to 224x224 - and want to use aug_transforms() to get some visual variation in images from cameratraps.  
This means that at each pass of the dls to the fine tuning system the model will see slightly different images so it learns how to detect with some variation.

In [ ]:
%%time
print('==== Loading images into tensors ====\n')
from sklearn.model_selection import train_test_split

path = Path('/kaggle/working/images')

# Stratified split to balance classes in train/valid sets
def stratified_splitter(items):
    labels = [parent_label(i) for i in items]
    train_idx, valid_idx = train_test_split(
        range(len(items)),
        test_size=0.2,
        stratify=labels,
        random_state=42
    )
    return train_idx, valid_idx

# Custom image trimming class
class CropTopBottom(Transform):
    def __init__(self, top_pct=0.1, bottom_pct=0.1):
        self.top_pct = top_pct
        self.bottom_pct = bottom_pct

    def encodes(self, img: PILImage):
        w, h = img.size
        top = int(h * self.top_pct)
        bottom = int(h * (1 - self.bottom_pct))
        return img.crop((0, top, w, bottom))

# DataBlock with custom splitter - removed batch transforms because the center zoom was cropping animals
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    splitter=stratified_splitter,
    item_tfms=[CropTopBottom(top_pct=0.1, bottom_pct=0.1), Resize(224, method='squish')],
    batch_tfms=aug_transforms()
)

# Build DataLoaders
dls = dblock.dataloaders(path)

# Show sample batch
dls.show_batch(max_n=9)

Now when resizing we aren't cutting off animals not centered and aren't introducing new zero padded areas.

# Fine tuning pre-trained models to our classifications
Initially we downloaded the resnet 18 model and fine tuned with 5 epochs to our dataset.  
Images , tests, and data are organized in folders and loaded into dls as appropriate.  

# Seeing what models are best
https://www.kaggle.com/code/morescope/which-image-models-are-best/

In [ ]:
import timm
timm.list_models('convnext*')

In [ ]:
%%time
learn = vision_learner(dls, 'convnext_tiny', metrics=error_rate)

if torch.cuda.is_available():
    learn.model = learn.model.cuda()
    print("Using:", next(learn.model.parameters()).device)  # should print 'cuda:0'
else:
    learn.model = learn.model.cpu()
    print("Using CPU")

# For this version we're trying 10 epochs
learn.fine_tune(5)

We went from 78 to 92% accuracy switching from resnet18 to convnext_tiny

## Confusion Matrix

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

Considerably better error rates on our dog, duck, and sandpiper categories

## Testing a single image prediction
Here we'll look at our test for a beaver and see if the model can predict it correctly.

In [ ]:
image_path = '/kaggle/working/_tests/Castor_canadensis.JPG'
pred, pred_idx, probs = learn.predict(PILImage.create(image_path))


PILImage.create(image_path).show(title=f"Probably: {pred} ({probs[pred_idx]:.2f})")

## Tabular predictions for all images in the folder

In [ ]:
# Predicting in a table for each image
test_path = Path("/kaggle/working/_tests")
test_files = get_image_files(test_path)

results = []
for f in test_files:
    pred_class, pred_idx, pred_probs = learn.predict(f)
    results.append({
        'file': f.name,
        'pred_class': str(pred_class),
        'probability': float(pred_probs[pred_idx]),
        'top3': [
            (learn.dls.vocab[i], format(float(pred_probs[i]), ".4f"))
            for i in pred_probs.argsort(descending=True)[:3]
        ]
    })

results_df = pd.DataFrame(results)
display(results_df)

## A pretty prediction plot for 9 of the test images

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
axes = axes.flatten()

for ax, img_file in zip(axes, test_files):
    img = PILImage.create(img_file)
    pred_class, pred_idx, pred_probs = learn.predict(img)

    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"Picture of: {img_file.name}\nPredicted: {pred_class} ({pred_probs[pred_idx]:.1%})", fontsize=9)

# Hide unused subplots if fewer than 9 images
for ax in axes[len(test_files):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

# Exporting the fine tuned model

In [ ]:
# export the model for futher use
learn.export('/kaggle/working/2025-06-18-convnet_tinyx25p-v10.pkl')